In [ ]:
# import pandas as pd
# import numpy as np
# import re

# # 1. Read log.csv and fix malformed lines
# cleaned_rows = []
# header = None

# with open('log.csv', 'r') as f:
#     lines = f.readlines()

# for line in lines:
#     line = line.strip()
#     if not line:
#         continue
    
#     parts = line.split(',')
    
#     if header is None:
#         header = parts
#         cleaned_rows.append(header)
#         continue
        
#     if len(parts) == 6:
#         cleaned_rows.append(parts)
#     elif len(parts) > 6:
#         # Handle the merged lines (e.g., ...gemini_2.5_flash214...)
#         first_row = parts[:6]
#         second_row = parts[5:] 
#         merged_val = parts[5]
        
#         # Split 'gemini_2.5_flash' and the following ID (e.g., '214')
#         match = re.match(r'(gemini_2\.5_flash)(\d+)', merged_val)
#         if match:
#             first_row[5] = match.group(1)
#             cleaned_rows.append(first_row)
            
#             second_row[0] = match.group(2)
#             cleaned_rows.append(second_row)

# # Create DataFrame
# log_df = pd.DataFrame(cleaned_rows[1:], columns=cleaned_rows[0])

# # Convert columns to numeric
# cols_to_numeric = ['id', 'input_token', 'output_token', 'thought_token']
# for col in cols_to_numeric:
#     log_df[col] = pd.to_numeric(log_df[col], errors='coerce')

# # 2. Filter Processed IDs (exclude where input, output, and thought are ALL 0)
# processed_df = log_df[
#     (log_df['input_token'] != 0) | 
#     (log_df['output_token'] != 0) | 
#     (log_df['thought_token'] != 0)
# ]

# processed_ids = sorted(processed_df['id'].unique())
# count = len(processed_ids)
# print(f"Number of processed IDs: {count}")

# # 3. Generate 100 random IDs
# np.random.seed(42)
# sampled_ids = np.random.choice(processed_ids, 100, replace=False)

# # 4. Process Manifest and create final CSV
# manifest_df = pd.read_csv('../../data/youtube_video_manifest.csv')
# manifest_df['id'] = pd.to_numeric(manifest_df['id'], errors='coerce')

# # Filter for the sampled IDs and select columns
# final_df = manifest_df[manifest_df['id'].isin(sampled_ids)].copy()
# final_df = final_df[['id', 'URL', 'Duration (sec)']]

# final_df.to_csv('sampled_processed_videos.csv', index=False)

Number of processed IDs: 167


In [6]:
import pandas as pd
import numpy as np
import re

# ... (Previous cleaning logic for log.csv) ...

# Create DataFrame
log_df = pd.DataFrame(cleaned_rows[1:], columns=cleaned_rows[0])

# Convert columns to numeric
numeric_cols = ['id', 'input_token', 'output_token', 'thought_token', 'execution_time_sec']
for col in numeric_cols:
    log_df[col] = pd.to_numeric(log_df[col], errors='coerce')

# Filter Processed IDs (exclude where input, output, and thought are ALL 0)
processed_df = log_df[
    (log_df['input_token'] != 0) | 
    (log_df['output_token'] != 0) | 
    (log_df['thought_token'] != 0)
].copy()

# Deduplicate to ensure unique IDs (keep last occurrence)
processed_df = processed_df.drop_duplicates(subset=['id'], keep='last')

# Generate 100 random IDs
processed_ids = processed_df['id'].unique()
np.random.seed(42)
if len(processed_ids) >= 100:
    sampled_ids = np.random.choice(processed_ids, 100, replace=False)
else:
    sampled_ids = processed_ids

# Filter for sampled IDs
sampled_log_df = processed_df[processed_df['id'].isin(sampled_ids)].copy()

# Load Manifest
manifest_df = pd.read_csv('../../data/youtube_video_manifest.csv')
manifest_df['id'] = pd.to_numeric(manifest_df['id'], errors='coerce')

# Merge Data
merged_df = pd.merge(sampled_log_df, manifest_df, on='id', how='left')

# Select final columns
final_df = merged_df[['id', 'URL', 'Duration (sec)', 'input_token', 'output_token', 'execution_time_sec']]

# Save CSV
final_df.to_csv('sampled_processed_videos.csv', index=False)

In [7]:
import os
import shutil
import pandas as pd

def create_sample_dataset():
    # Configuration
    input_csv = 'sampled_processed_videos.csv'
    source_dirs = {'csv': 'csv', 'txt': 'txt'}
    base_output_dir = 'sample'
    
    # 1. Read the sampled IDs
    try:
        df = pd.read_csv(input_csv)
        # Ensure 'id' is treated as a string to match filenames (e.g., "188.csv")
        ids = df['id'].astype(str).tolist()
        print(f"Loaded {len(ids)} IDs from {input_csv}")
    except FileNotFoundError:
        print(f"Error: Could not find {input_csv}. Make sure it exists in this folder.")
        return

    # 2. Create Destination Directories
    for subfolder in source_dirs.keys():
        dest_path = os.path.join(base_output_dir, subfolder)
        os.makedirs(dest_path, exist_ok=True)
        print(f"Created directory: {dest_path}")

    # 3. Copy Files
    success_count = 0
    missing_files = []

    print("\nStarting file copy process...")
    
    for file_id in ids:
        # We need to copy both the csv and txt version for this ID
        for file_type, src_folder in source_dirs.items():
            filename = f"{file_id}.{file_type}"
            
            src_path = os.path.join(src_folder, filename)
            dst_path = os.path.join(base_output_dir, file_type, filename)
            
            if os.path.exists(src_path):
                shutil.copy2(src_path, dst_path)
            else:
                missing_files.append(src_path)
        
        success_count += 1

    # 4. Final Report
    print(f"\nProcess complete!")
    print(f"Processed {success_count} IDs.")
    
    if missing_files:
        print(f"\nWarning: {len(missing_files)} files were not found in the source folders:")
        for missing in missing_files[:10]: # Print first 10 missing
            print(f" - {missing}")
        if len(missing_files) > 10:
            print(f" ... and {len(missing_files) - 10} more.")
    else:
        print("All files copied successfully.")

if __name__ == "__main__":
    create_sample_dataset()

Loaded 100 IDs from sampled_processed_videos.csv
Created directory: sample/csv
Created directory: sample/txt

Starting file copy process...

Process complete!
Processed 100 IDs.
All files copied successfully.
